# Final Report (Lightweight)

Use this notebook to inspect results produced by the core pipeline.

- Core output notebook: `final_core_output.ipynb`
- Core run log: `core_run.log`
- Backtest cache: `../outputs/cache/backtest_I*.pkl`


In [ ]:
from pathlib import Path
import pandas as pd

base = Path('.').resolve()
log_path = base / 'core_run.log'
out_nb = base / 'final_core_output.ipynb'
cache_dir = (base.parent / 'outputs' / 'cache').resolve()

print('core output exists:', out_nb.exists(), out_nb)
print('core log exists   :', log_path.exists(), log_path)
print('cache dir         :', cache_dir)

for p in sorted(cache_dir.glob('backtest_I*.pkl')):
    print('-', p.name, f'({p.stat().st_size/1024/1024:.1f} MB)')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cache_dir = (Path('.').resolve().parent / 'outputs' / 'cache')
series = []
for h in [5, 20, 60]:
    p = cache_dir / f'backtest_I{h}.pkl'
    if p.exists():
        df = pd.read_pickle(p)
        col = 'CNN_LS_net' if 'CNN_LS_net' in df.columns else ('CNN_LS_net_EW' if 'CNN_LS_net_EW' in df.columns else None)
        if col:
            s = np.log1p(df[col].fillna(0)).cumsum()
            s.name = f'I{h}'
            series.append(s)

if not series:
    print('No backtest cache found yet.')
else:
    plt.figure(figsize=(10, 5))
    for s in series:
        plt.plot(s.index, s.values, label=s.name)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.title('Cumulative Log Return from Cached Backtests')
    plt.tight_layout()
    plt.show()
